# 01 Bronze ingest

Raw volume JSON → **schema-enforced** Delta tables. Bronze layer keeps the data as parsed
(no cleaning); it only enforces a schema and makes re-ingestion idempotent.

- **`bronze_battles`**: one row per `battle_id`, loaded with an idempotent `MERGE`.
  The same battle appears in two players' logs, so the `MERGE` collapses those to one row.
- **`bronze_cards`**: small, slowly-changing dim; full-refresh overwrite each run.
- **`bronze_players`**: the crawl seed, kept for lineage (overwrite).

## Config

In [0]:
from pyspark.sql import functions as F, Window, types as T
from delta.tables import DeltaTable

CATALOG, SCHEMA = "workspace", "clash"
RAW = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

BRONZE_BATTLES = f"{CATALOG}.{SCHEMA}.bronze_battles"
BRONZE_CARDS   = f"{CATALOG}.{SCHEMA}.bronze_cards"
BRONZE_PLAYERS = f"{CATALOG}.{SCHEMA}.bronze_players"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

DataFrame[]

## Bronze battles
Schema enforcement, deduplication, and idempotent merge are applied in the bronze battlelog layer.

In [0]:
battles_schema = T.StructType([
    T.StructField("battle_id", T.StringType()),
    T.StructField("battle_time", T.StringType()),
    T.StructField("battle_time_raw", T.StringType()),
    T.StructField("type", T.StringType()),
    T.StructField("is_ladder", T.BooleanType()),
    T.StructField("game_mode", T.StringType()),
    T.StructField("arena_id", T.LongType()),
    T.StructField("arena_name", T.StringType()),
    T.StructField("player_tag", T.StringType()),
    T.StructField("player_name", T.StringType()),
    T.StructField("player_starting_trophies", T.LongType()),
    T.StructField("player_trophy_change", T.LongType()),
    T.StructField("player_crowns", T.LongType()),
    T.StructField("player_cards", T.ArrayType(T.LongType())),
    T.StructField("player_deck_hash", T.StringType()),
    T.StructField("opponent_tag", T.StringType()),
    T.StructField("opponent_name", T.StringType()),
    T.StructField("opponent_starting_trophies", T.LongType()),
    T.StructField("opponent_trophy_change", T.LongType()),
    T.StructField("opponent_crowns", T.LongType()),
    T.StructField("opponent_cards", T.ArrayType(T.LongType())),
    T.StructField("opponent_deck_hash", T.StringType()),
    T.StructField("win", T.BooleanType()),
    T.StructField("source_player_tag", T.StringType()),
    T.StructField("ingested_at", T.StringType()),
])

raw = spark.read.schema(battles_schema).json(f"{RAW}/*/battles.jsonl")

In [0]:
# Deduplicate within this load so each battle_id appears once before the MERGE.
# Always keep the row with the lowest source_player_tag for a battle, so the kept row is deterministic across reruns.
w = Window.partitionBy("battle_id").orderBy("source_player_tag")
staged = (raw
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1").drop("rn")
    .withColumn("bronze_loaded_at", F.current_timestamp()))

print(raw.count(), "raw rows ->", staged.count(), "unique battle_id")

319879 raw rows -> 318176 unique battle_id


### Idempotent MERGE

First run creates the Delta table; later runs `MERGE` on `battle_id` and insert only battles not already there. This ensures re-running the pipeline does not create duplicates or overwrite existing battles.

In [0]:
if not spark.catalog.tableExists(BRONZE_BATTLES):
    staged.write.format("delta").saveAsTable(BRONZE_BATTLES)
    print("created", BRONZE_BATTLES)
else:
    (DeltaTable.forName(spark, BRONZE_BATTLES).alias("t")
        .merge(
            source=staged.alias("s"),
            condition="t.battle_id = s.battle_id"
        ).whenNotMatchedInsertAll()
        .execute())
    print("merged into", BRONZE_BATTLES)

merged into workspace.clash.bronze_battles


In [0]:
bb = spark.table(BRONZE_BATTLES)
print(bb.count(), "rows;", bb.select("battle_id").distinct().count(), "distinct battle_id")
display(bb.limit(10))

603439 rows; 603439 distinct battle_id


battle_id,battle_time,battle_time_raw,type,is_ladder,game_mode,arena_id,arena_name,player_tag,player_name,player_starting_trophies,player_trophy_change,player_crowns,player_cards,player_deck_hash,opponent_tag,opponent_name,opponent_starting_trophies,opponent_trophy_change,opponent_crowns,opponent_cards,opponent_deck_hash,win,source_player_tag,ingested_at,bronze_loaded_at
0001f6cccad0e9482c7f3988a52b6e8ce0291e9f,2026-06-06T02:19:03+00:00,20260606T021903.000Z,trail,false,TripleElixir_Friendly,54000152,Legendary Arena,#2GLUUVYY0,Helpzv,5,null,1,"List(26000045, 26000103, 28000017, 26000046, 26000050, 26000025, 28000007, 26000021)",9849c76134c7151798017247972c65a32bc64e7b,#908Q2VUY,Rollin_Uhpp,5,1,2,"List(26000056, 27000001, 27000010, 26000054, 26000020, 28000013, 28000012, 26000080)",b4a09230a372e9826c1d20f44272e1f744858b04,false,#2GLUUVYY0,2026-06-08T03:17:11.025146+00:00,2026-06-09T19:43:41.138Z
000215fbbc4baffadba28de50414f2d2b79d8d42,2026-06-06T02:57:48+00:00,20260606T025748.000Z,trail,true,Ladder,168000147,Seasonal Arena I,#P8099RLJ,SOSA,14211,30,2,"List(26000011, 26000017, 26000012, 28000008, 26000043, 26000016, 28000006, 28000001)",20794aad1fed4bddfbf1157afc9d3bfa59564975,#9J0PGJ2,Shamah,14211,-30,0,"List(26000047, 26000038, 26000011, 26000021, 28000017, 26000080, 28000007, 26000001)",cabce2ffcf7af2cc7f407ea8f43a02599fbb8bba,true,#P8099RLJ,2026-06-08T03:47:09.735415+00:00,2026-06-09T19:43:41.138Z
00046b564e8cf1abc4d53c73c337f5a37d2bdd42,2026-06-02T16:21:55+00:00,20260602T162155.000Z,trail,true,Ladder,168000148,Seasonal Arena II,#YJLV88CYU,☆XÑ ØØD☆,14500,31,1,"List(28000017, 26000038, 26000010, 26000032, 27000004, 26000014, 28000015, 26000006)",4850b4bf8ef2167ef183c07de0b12e340a7abd21,#VRYRYGRGR,Crystal_Neko,14513,-13,0,"List(26000007, 26000003, 26000017, 28000005, 26000037, 28000001, 26000012, 28000000)",16b83f8c6225088ea913147d52d7dade0a0ffb14,true,#YJLV88CYU,2026-06-08T02:50:44.283920+00:00,2026-06-09T19:43:41.138Z
00046cbf9fb85a6cb788bed18621a769964a877b,2026-06-07T19:32:47+00:00,20260607T193247.000Z,trail,true,Ladder,168000147,Seasonal Arena I,#JP9YUQ82,lalonc,14332,30,3,"List(26000055, 26000039, 26000037, 28000007, 28000001, 26000018, 26000021, 26000014)",784750dfed15772babdd2ceb3ab0a7324ca43995,#9RC8PURG,Sadullah,14332,-30,1,"List(26000055, 26000018, 26000014, 26000030, 27000001, 26000021, 26000010, 28000011)",198beb554330e7ca5cdc037c7410353aaaffe2aa,true,#JP9YUQ82,2026-06-08T03:29:18.103394+00:00,2026-06-09T19:43:41.138Z
0004849ceee06cfef172f6caaf1bee627330c61d,2026-06-07T07:49:59+00:00,20260607T074959.000Z,trail,true,Ladder,168000147,Seasonal Arena I,#P80L0YU00,CHIRIPA10,14241,-30,0,"List(26000055, 26000072, 26000049, 26000016, 26000058, 26000030, 26000041, 28000001)",c0396b73f5ea4a0324f3c4a7b8effa126bf56a3b,#UR0VR0C8C,free palestine�,14241,30,1,"List(26000035, 26000006, 27000012, 28000001, 26000057, 26000047, 28000000, 26000030)",691915890fc7954cb922848bb3dde8c1d22fd268,false,#P80L0YU00,2026-06-08T03:24:18.736160+00:00,2026-06-09T19:43:41.138Z
00051035097d26810bca7220fa057a02b4e950bc,2026-06-07T18:34:39+00:00,20260607T183439.000Z,trail,true,Ladder,168000147,Seasonal Arena I,#2PPG0RPGU,Germen Supremo,14480,30,1,"List(26000011, 26000039, 26000006, 26000029, 26000005, 28000000, 28000008, 27000009)",c9f3c89f45dae93b45741ea0bec0637bee3ce1bc,#8QJ2CGLPG,GRECOX,14480,-30,0,"List(26000047, 26000062, 26000045, 28000001, 26000033, 26000034, 26000042, 26000021)",f4357ddea3cdccc97fcd60e65001a7e82b344a57,true,#2PPG0RPGU,2026-06-08T02:44:36.181193+00:00,2026-06-09T19:43:41.138Z
00074f92dbd6d0c0eef129ab64ea07b073aca252,2026-05-17T02:02:51+00:00,20260517T020251.000Z,boatBattle,false,ClanWar_BoatBattle,54000046,Legendary Arena,#GCGRJJV29,XxFabrixX,null,null,0,"List(26000035, 26000045, 26000021, 26000051, 26000007, 26000004, 26000006, 26000033, 26000018, 26000022, 26000055, 26000085)",63912913dc4e6e7b7ede53a6683bc507108e86f5,#CCY2G0JY0,Askvt,11560,null,3,"List(26000017, 26000011, 26000012, 26000018, 28000001, 26000041, 26000021, 2600

## Bronze cards

Loads card dimension data into a schema-enforced Delta table (full overwrite).

In [0]:
cards_schema = T.StructType([
    T.StructField("card_id", T.LongType()),
    T.StructField("name", T.StringType()),
    T.StructField("rarity", T.StringType()),
    T.StructField("elixir_cost", T.LongType()),
    T.StructField("max_level", T.LongType()),
])
cards = (spark.read.schema(cards_schema).option("multiline", "true")
    .json(f"{RAW}/cards.json")
    .withColumn("bronze_loaded_at", F.current_timestamp()))
cards.write.format("delta").mode("overwrite").saveAsTable(BRONZE_CARDS)
print(spark.table(BRONZE_CARDS).count(), "cards")
display(spark.table(BRONZE_CARDS).orderBy("card_id").limit(10))

121 cards


card_id,name,rarity,elixir_cost,max_level,bronze_loaded_at
26000000,Knight,common,3,16,2026-06-09T19:44:04.287Z
26000001,Archers,common,3,16,2026-06-09T19:44:04.287Z
26000002,Goblins,common,2,16,2026-06-09T19:44:04.287Z
26000003,Giant,rare,5,14,2026-06-09T19:44:04.287Z
26000004,P.E.K.K.A,epic,7,11,2026-06-09T19:44:04.287Z
26000005,Minions,common,3,16,2026-06-09T19:44:04.287Z
26000006,Balloon,epic,5,11,2026-06-09T19:44:04.287Z
26000007,Witch,epic,5,11,2026-06-09T19:44:04.287Z
26000008,Barbarians,common,5,16,2026-06-09T19:44:04.287Z
26000009,Golem,epic,8,11,2026-06-09T19:44:04.287Z


## Bronze players (seed lineage)

Not an analytical table, just a record of the crawl seed for player battlelogs (overwrite)

In [0]:
players_schema = T.StructType([
    T.StructField("tag", T.StringType()),
    T.StructField("name", T.StringType()),
    T.StructField("trophies", T.LongType()),
    T.StructField("clan_tag", T.StringType()),
])
players = (spark.read.schema(players_schema).option("multiline", "true")
    .json(f"{RAW}/players.json")
    .withColumn("bronze_loaded_at", F.current_timestamp()))
players.write.format("delta").mode("overwrite").saveAsTable(BRONZE_PLAYERS)
print(spark.table(BRONZE_PLAYERS).count(), "players")
display(spark.table(BRONZE_PLAYERS).limit(10))

9976 players


tag,name,trophies,clan_tag,bronze_loaded_at
#PPVL99U0R,flazrael-antifa,14000,#QRCCY0LJ,2026-06-09T19:44:10.632Z
#V9QC0UYVG,gui,14000,#QRCCY0LJ,2026-06-09T19:44:10.632Z
#GYGLCC28,dtrem7複|,14000,#QRCCY0LJ,2026-06-09T19:44:10.632Z
#9J9UQYJ82,Łʑ☮ɴąɴɗѳ,14000,#QRCCY0LJ,2026-06-09T19:44:10.632Z
#QJ9J2YQ2,Robert,14000,#QRCCY0LJ,2026-06-09T19:44:10.632Z
#8PU88JV0G,LJira¥a,14000,#QRCCY0LJ,2026-06-09T19:44:10.632Z
#QQ2UGVV0L,TEF丨Jerry ✨ CR,14000,#QRCCY0LJ,2026-06-09T19:44:10.632Z
#PCQCJRYYR,Jhonata Br,14000,#QRCCY0LJ,2026-06-09T19:44:10.632Z
#90QLQUG09,Dodowq,14000,#QRCCY0LJ,2026-06-09T19:44:10.632Z
#L9GQJ9PUY,#1 ✨ Storm │優速⚡,14000,#QRCCY0LJ,2026-06-09T19:44:10.632Z


## Done

Bronze is loaded and idempotent. Next: `02_silver_transform`.